## Example 1: Basic RDF Graph Operations

In [ ]:
# Example 1: Basic RDF graph creation and triple management
print("Example 1: Basic RDF graph creation and triple management")

try:
    import rdflib
    from rdflib import Graph, Namespace, RDF, RDFS, Literal, URIRef, BNode
    
    print(f"RDFLib version: {rdflib.__version__}")
    
    # Create a new RDF graph
    g = Graph()
    print(f"Created empty RDF graph: {type(g)}")
    
    # Define namespaces for better URI management
    ex = Namespace("http://example.org/")
    foaf = Namespace("http://xmlns.com/foaf/0.1/")
    
    # Bind prefixes to the graph for cleaner serialization
    g.bind("ex", ex)
    g.bind("foaf", foaf)
    
    # Create URIs for resources
    john = ex.john
    mary = ex.mary
    
    # Add triples to the graph (subject, predicate, object)
    g.add((john, RDF.type, foaf.Person))
    g.add((john, foaf.name, Literal("John Smith")))
    g.add((john, foaf.age, Literal(32)))
    g.add((john, foaf.knows, mary))
    
    g.add((mary, RDF.type, foaf.Person))
    g.add((mary, foaf.name, Literal("Mary Johnson")))
    g.add((mary, foaf.age, Literal(28)))
    
    print(f"Graph now contains {len(g)} triples")
    
    # Query the graph using simple iteration
    print("\nPeople in the graph:")
    for person in g.subjects(RDF.type, foaf.Person):
        name = g.value(person, foaf.name)
        age = g.value(person, foaf.age)
        print(f"  - {name} (age {age})")
    
    # Check relationships
    if (john, foaf.knows, mary) in g:
        print(f"\n{g.value(john, foaf.name)} knows {g.value(mary, foaf.name)}")
    
    # Demonstrate triple patterns
    print("\nAll name relationships:")
    for subj, pred, obj in g.triples((None, foaf.name, None)):
        print(f"  {subj} has name '{obj}'")
    
    # Basic graph statistics
    print(f"\nGraph statistics:")
    print(f"  Total triples: {len(g)}")
    print(f"  Total subjects: {len(set(g.subjects()))}")
    print(f"  Total predicates: {len(set(g.predicates()))}")
    print(f"  Total objects: {len(set(g.objects()))}")
    
    print("Basic RDF graph operations completed successfully")
    
except Exception as e:
    print(f"Error in RDF graph creation: {e}")
    raise AssertionError(
        f"rdflib example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: SPARQL Queries and Data Retrieval

In [ ]:
# Example 2: SPARQL queries and RDF data retrieval
print("Example 2: SPARQL queries and RDF data retrieval")

try:
    from rdflib import Graph, Namespace, RDF, RDFS, FOAF, Literal, URIRef
    
    # Create and populate a graph with organizational data
    g = Graph()
    ex = Namespace("http://example.org/")
    g.bind("ex", ex)
    g.bind("foaf", FOAF)
    
    # Add organizational data
    companies = [
        ("ex:TechCorp", "TechCorp Inc.", "Technology"),
        ("ex:DataSystems", "Data Systems LLC", "Software"),
        ("ex:WebSolutions", "Web Solutions Co.", "Web Development")
    ]
    
    employees = [
        ("ex:alice", "Alice Brown", "ex:TechCorp", "Developer", 75000),
        ("ex:bob", "Bob Wilson", "ex:TechCorp", "Manager", 95000),
        ("ex:carol", "Carol Davis", "ex:DataSystems", "Analyst", 65000),
        ("ex:david", "David Miller", "ex:WebSolutions", "Designer", 60000)
    ]
    
    # Define custom properties
    worksFor = ex.worksFor
    hasRole = ex.hasRole
    hasSalary = ex.hasSalary
    companyName = ex.companyName
    industry = ex.industry
    
    # Add company data
    for company_uri, name, ind in companies:
        company = URIRef(company_uri.replace("ex:", str(ex)))
        g.add((company, RDF.type, ex.Company))
        g.add((company, companyName, Literal(name)))
        g.add((company, industry, Literal(ind)))
    
    # Add employee data
    for emp_uri, name, comp_uri, role, salary in employees:
        employee = URIRef(emp_uri.replace("ex:", str(ex)))
        company = URIRef(comp_uri.replace("ex:", str(ex)))
        
        g.add((employee, RDF.type, FOAF.Person))
        g.add((employee, FOAF.name, Literal(name)))
        g.add((employee, worksFor, company))
        g.add((employee, hasRole, Literal(role)))
        g.add((employee, hasSalary, Literal(salary)))
    
    print(f"Created organizational graph with {len(g)} triples")
    
    # SPARQL Query 1: Find all employees and their companies
    print("\n=== SPARQL Query 1: Employees and Companies ===")
    query1 = """
        PREFIX ex: <http://example.org/>
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
        
        SELECT ?name ?companyName ?role
        WHERE {
            ?person foaf:name ?name ;
                   ex:worksFor ?company ;
                   ex:hasRole ?role .
            ?company ex:companyName ?companyName .
        }
        ORDER BY ?companyName ?name
    """
    
    results1 = g.query(query1)
    for row in results1:
        print(f"  {row.name} works at {row.companyName} as {row.role}")
    
    # SPARQL Query 2: Find employees with salary above threshold
    print("\n=== SPARQL Query 2: High Salary Employees ===")
    query2 = """
        PREFIX ex: <http://example.org/>
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
        
        SELECT ?name ?salary ?role
        WHERE {
            ?person foaf:name ?name ;
                   ex:hasSalary ?salary ;
                   ex:hasRole ?role .
            FILTER (?salary > 70000)
        }
        ORDER BY DESC(?salary)
    """
    
    results2 = g.query(query2)
    for row in results2:
        print(f"  {row.name}: ${row.salary} ({row.role})")
    
    # SPARQL Query 3: Aggregate data by company
    print("\n=== SPARQL Query 3: Company Statistics ===")
    query3 = """
        PREFIX ex: <http://example.org/>
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
        
        SELECT ?companyName (COUNT(?person) AS ?employeeCount) (AVG(?salary) AS ?avgSalary)
        WHERE {
            ?person foaf:name ?name ;
                   ex:worksFor ?company ;
                   ex:hasSalary ?salary .
            ?company ex:companyName ?companyName .
        }
        GROUP BY ?companyName
        ORDER BY DESC(?avgSalary)
    """
    
    results3 = g.query(query3)
    for row in results3:
        print(f"  {row.companyName}: {row.employeeCount} employees, avg salary ${int(row.avgSalary)}")
    
    # SPARQL Query 4: Find specific patterns
    print("\n=== SPARQL Query 4: Technology Companies ===")
    query4 = """
        PREFIX ex: <http://example.org/>
        
        SELECT ?companyName ?industry
        WHERE {
            ?company ex:companyName ?companyName ;
                    ex:industry ?industry .
            FILTER (CONTAINS(LCASE(?industry), "tech"))
        }
    """
    
    results4 = g.query(query4)
    for row in results4:
        print(f"  {row.companyName} - Industry: {row.industry}")
    
    print("\nSPARQL query operations completed successfully")
    
except Exception as e:
    print(f"Error in SPARQL queries: {e}")
    raise AssertionError(
        f"rdflib example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: RDF Serialization and Format Conversion

In [ ]:
# Example 3: RDF serialization and format conversion
print("Example 3: RDF serialization and format conversion")

from rdflib import Graph, Namespace, RDF, FOAF, Literal
from rdflib.namespace import DC

# Create a knowledge base about books
g = Graph()
ex = Namespace("http://example.org/books/")
g.bind("ex", ex)

# Add book and author data
author1 = ex.author1
book1 = ex.book1

g.add((author1, RDF.type, FOAF.Person))
g.add((author1, FOAF.name, Literal("Jane Austen")))
g.add((book1, RDF.type, ex.Book))
g.add((book1, DC.title, Literal("Pride and Prejudice")))
g.add((book1, DC.creator, author1))

print(f"Created graph with {len(g)} triples\n")

# Serialize to different formats (returns str in rdflib 7.x)
turtle_data = g.serialize(format='turtle')
print(f"Turtle format ({len(turtle_data)} chars):\n{turtle_data[:200]}...\n")

# Test round-trip: serialize and parse back
g2 = Graph()
g2.parse(data=turtle_data, format='turtle')
print(f"Round-trip: {len(g)} -> {len(g2)} triples, Success: {len(g) == len(g2)}")

## Example 4: Ontologies and Semantic Reasoning

In [ ]:
# Example 4: Ontologies, inference, and semantic reasoning
print("Example 4: Ontologies, inference, and semantic reasoning")

try:
    from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef
    # rdflib_to_networkx_multidigraph needs networkx, which rdflib does not install:
    # pip install networkx
    from rdflib.extras.external_graph_libs import rdflib_to_networkx_multidigraph
    
    # Create an ontological graph with class hierarchies and properties
    g = Graph()
    ex = Namespace("http://example.org/animals/")
    g.bind("ex", ex)
    g.bind("rdfs", RDFS)
    g.bind("owl", OWL)
    
    # Define class hierarchy
    animal = ex.Animal
    mammal = ex.Mammal
    bird = ex.Bird
    dog = ex.Dog
    cat = ex.Cat
    eagle = ex.Eagle
    
    # Class definitions and hierarchy
    g.add((animal, RDF.type, OWL.Class))
    g.add((mammal, RDF.type, OWL.Class))
    g.add((bird, RDF.type, OWL.Class))
    g.add((dog, RDF.type, OWL.Class))
    g.add((cat, RDF.type, OWL.Class))
    g.add((eagle, RDF.type, OWL.Class))
    
    # Subclass relationships
    g.add((mammal, RDFS.subClassOf, animal))
    g.add((bird, RDFS.subClassOf, animal))
    g.add((dog, RDFS.subClassOf, mammal))
    g.add((cat, RDFS.subClassOf, mammal))
    g.add((eagle, RDFS.subClassOf, bird))
    
    # Define properties
    hasColor = ex.hasColor
    canFly = ex.canFly
    isEndangered = ex.isEndangered
    hasHabitat = ex.hasHabitat
    
    g.add((hasColor, RDF.type, OWL.DatatypeProperty))
    g.add((canFly, RDF.type, OWL.DatatypeProperty))
    g.add((isEndangered, RDF.type, OWL.DatatypeProperty))
    g.add((hasHabitat, RDF.type, OWL.DatatypeProperty))
    
    # Create individual instances
    buddy = ex.buddy  # a dog
    fluffy = ex.fluffy  # a cat
    sam = ex.sam  # an eagle
    
    # Instance data
    g.add((buddy, RDF.type, dog))
    g.add((buddy, hasColor, Literal("brown")))
    g.add((buddy, canFly, Literal(False)))
    g.add((buddy, hasHabitat, Literal("domesticated")))
    
    g.add((fluffy, RDF.type, cat))
    g.add((fluffy, hasColor, Literal("white")))
    g.add((fluffy, canFly, Literal(False)))
    g.add((fluffy, hasHabitat, Literal("domesticated")))
    
    g.add((sam, RDF.type, eagle))
    g.add((sam, hasColor, Literal("brown")))
    g.add((sam, canFly, Literal(True)))
    g.add((sam, isEndangered, Literal(False)))
    g.add((sam, hasHabitat, Literal("mountains")))
    
    print(f"Created ontological graph with {len(g)} triples")
    
    # Query class hierarchy
    print("\n=== Class Hierarchy Analysis ===")
    hierarchy_query = """
        PREFIX ex: <http://example.org/animals/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        
        SELECT ?subClass ?superClass
        WHERE {
            ?subClass rdfs:subClassOf ?superClass .
        }
        ORDER BY ?superClass ?subClass
    """
    
    for row in g.query(hierarchy_query):
        subclass_name = str(row.subClass).split('/')[-1]
        superclass_name = str(row.superClass).split('/')[-1]
        print(f"  {subclass_name} is a subclass of {superclass_name}")
    
    # Infer types through class hierarchy
    print("\n=== Type Inference ===")
    inference_query = """
        PREFIX ex: <http://example.org/animals/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT ?individual ?directType ?inferredType
        WHERE {
            ?individual rdf:type ?directType .
            ?directType rdfs:subClassOf+ ?inferredType .
            FILTER(?directType != ?inferredType)
        }
        ORDER BY ?individual
    """
    
    results = list(g.query(inference_query))
    if results:
        for row in results:
            ind_name = str(row.individual).split('/')[-1]
            direct_name = str(row.directType).split('/')[-1]
            inferred_name = str(row.inferredType).split('/')[-1]
            print(f"  {ind_name} is directly a {direct_name}, inferred to be a {inferred_name}")
    else:
        print("  No transitive inferences found (requires reasoning engine)")
    
    # Property analysis
    print("\n=== Property Analysis ===")
    property_query = """
        PREFIX ex: <http://example.org/animals/>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT ?individual ?property ?value
        WHERE {
            ?individual ?property ?value .
            FILTER(?property != rdf:type)
        }
        ORDER BY ?individual ?property
    """
    
    for row in g.query(property_query):
        ind_name = str(row.individual).split('/')[-1]
        prop_name = str(row.property).split('/')[-1]
        print(f"  {ind_name} {prop_name}: {row.value}")
    
    # Semantic queries with reasoning-like patterns
    print("\n=== Semantic Queries ===")
    
    # Find all animals (direct and inferred)
    animal_query = """
        PREFIX ex: <http://example.org/animals/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT DISTINCT ?individual ?type
        WHERE {
            ?individual rdf:type ?type .
            ?type rdfs:subClassOf* ex:Animal .
        }
        ORDER BY ?individual
    """
    
    print("  All animals in the knowledge base:")
    for row in g.query(animal_query):
        ind_name = str(row.individual).split('/')[-1]
        type_name = str(row.type).split('/')[-1]
        print(f"    {ind_name} (type: {type_name})")
    
    # Find animals that can fly
    flying_query = """
        PREFIX ex: <http://example.org/animals/>
        
        SELECT ?individual ?color
        WHERE {
            ?individual ex:canFly true ;
                       ex:hasColor ?color .
        }
    """
    
    print("\n  Flying animals:")
    for row in g.query(flying_query):
        ind_name = str(row.individual).split('/')[-1]
        print(f"    {ind_name} (color: {row.color})")
    
    # Graph structure analysis
    print("\n=== Graph Structure ===")
    print(f"Total triples: {len(g)}")
    print(f"Total classes: {len(list(g.subjects(RDF.type, OWL.Class)))}")
    print(f"Total individuals: {len(set(g.subjects()) - set(g.subjects(RDF.type, OWL.Class)) - set(g.subjects(RDF.type, OWL.DatatypeProperty)))}")
    print(f"Total properties: {len(list(g.subjects(RDF.type, OWL.DatatypeProperty)))}")
    
    # Simple consistency check
    print("\n=== Basic Consistency Check ===")
    
    nx_graph = rdflib_to_networkx_multidigraph(g)
    assert (len(g), nx_graph.number_of_edges()) == (28, 28), (len(g), nx_graph.number_of_edges())
    print(f"NetworkX MultiDiGraph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")

    # Check for individuals with conflicting properties
    consistency_issues = []
    
    # Check if any non-flying animals claim to fly
    for individual in g.subjects(RDF.type, mammal):
        can_fly_vals = list(g.objects(individual, canFly))
        if can_fly_vals and can_fly_vals[0].value == True:
            consistency_issues.append(f"Mammal {individual} claims to fly")
    
    if consistency_issues:
        print("  Consistency issues found:")
        for issue in consistency_issues:
            print(f"    - {issue}")
    else:
        print("  No basic consistency issues detected")
    
    print("Ontological reasoning and semantic queries completed successfully")
    
except ImportError:
    raise
except Exception as e:
    print(f"Error in semantic reasoning: {e}")
    raise AssertionError(
        f"rdflib example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Advanced RDF Operations and Analytics

In [ ]:
# Example 5: Advanced RDF operations and graph analytics
print("Example 5: Advanced RDF operations and graph analytics")

try:
    from rdflib import Graph, ConjunctiveGraph, Namespace, RDF, RDFS, Literal, URIRef, BNode
    from rdflib.graph import DATASET_DEFAULT_GRAPH_ID
    import time
    
    # Create a conjunctive graph for named graph operations
    cg = ConjunctiveGraph()
    
    # Define namespaces
    ex = Namespace("http://example.org/")
    social = Namespace("http://example.org/social/")
    work = Namespace("http://example.org/work/")
    
    cg.bind("ex", ex)
    cg.bind("social", social)
    cg.bind("work", work)
    
    # Create named graphs for different contexts
    social_graph = Graph(store=cg.store, identifier=social.context)
    work_graph = Graph(store=cg.store, identifier=work.context)
    
    print("=== Named Graph Operations ===")
    
    # Add social network data
    alice = ex.alice
    bob = ex.bob
    charlie = ex.charlie
    diana = ex.diana
    
    # Social relationships
    social_graph.add((alice, social.knows, bob))
    social_graph.add((alice, social.knows, charlie))
    social_graph.add((bob, social.knows, diana))
    social_graph.add((charlie, social.knows, diana))
    social_graph.add((alice, social.likes, ex.python))
    social_graph.add((bob, social.likes, ex.javascript))
    social_graph.add((charlie, social.likes, ex.python))
    social_graph.add((diana, social.likes, ex.datascience))
    
    # Work relationships
    work_graph.add((alice, work.worksAt, ex.TechCorp))
    work_graph.add((bob, work.worksAt, ex.StartupInc))
    work_graph.add((charlie, work.worksAt, ex.TechCorp))
    work_graph.add((diana, work.worksAt, ex.DataCorp))
    work_graph.add((alice, work.hasRole, Literal("Senior Developer")))
    work_graph.add((bob, work.hasRole, Literal("Frontend Engineer")))
    work_graph.add((charlie, work.hasRole, Literal("Data Scientist")))
    work_graph.add((diana, work.hasRole, Literal("ML Engineer")))
    
    print(f"Social graph: {len(social_graph)} triples")
    print(f"Work graph: {len(work_graph)} triples")
    print(f"Total conjunctive graph: {len(cg)} triples")
    
    # Query across named graphs
    print("\n=== Cross-Graph Queries ===")
    cross_query = """
        PREFIX ex: <http://example.org/>
        PREFIX social: <http://example.org/social/>
        PREFIX work: <http://example.org/work/>
        
        SELECT ?person ?friend ?workplace ?friendWorkplace
        WHERE {
            GRAPH social: {
                ?person social:knows ?friend .
            }
            GRAPH work: {
                ?person work:worksAt ?workplace .
                ?friend work:worksAt ?friendWorkplace .
            }
        }
    """
    
    results = cg.query(cross_query)
    for row in results:
        person_name = str(row.person).split('/')[-1]
        friend_name = str(row.friend).split('/')[-1]
        workplace_name = str(row.workplace).split('/')[-1]
        friend_workplace = str(row.friendWorkplace).split('/')[-1]
        print(f"  {person_name} knows {friend_name} | {person_name}@{workplace_name}, {friend_name}@{friend_workplace}")
    
    # Graph analytics and metrics
    print("\n=== Graph Analytics ===")
    
    # Create a unified graph for analytics
    analytics_graph = Graph()
    for triple in social_graph:
        analytics_graph.add(triple)
    for triple in work_graph:
        analytics_graph.add(triple)
    
    # Calculate node degrees in social network
    people = set()
    for s, p, o in social_graph.triples((None, social.knows, None)):
        people.add(s)
        people.add(o)
    
    print("Social network analysis:")
    person_connections = {}
    for person in people:
        # Count outgoing connections
        outgoing = len(list(social_graph.objects(person, social.knows)))
        # Count incoming connections
        incoming = len(list(social_graph.subjects(social.knows, person)))
        total = outgoing + incoming
        person_connections[person] = {'out': outgoing, 'in': incoming, 'total': total}
        
        person_name = str(person).split('/')[-1]
        print(f"  {person_name}: {outgoing} outgoing, {incoming} incoming, {total} total connections")
    
    # Find most connected person
    most_connected = max(person_connections.items(), key=lambda x: x[1]['total'])
    most_connected_name = str(most_connected[0]).split('/')[-1]
    print(f"  Most connected: {most_connected_name} ({most_connected[1]['total']} connections)")
    
    # Analyze interests and workplace correlations
    print("\n=== Interest and Workplace Analysis ===")
    
    interest_workplace_query = """
        PREFIX ex: <http://example.org/>
        PREFIX social: <http://example.org/social/>
        PREFIX work: <http://example.org/work/>
        
        SELECT ?person ?interest ?workplace ?role
        WHERE {
            ?person social:likes ?interest ;
                   work:worksAt ?workplace ;
                   work:hasRole ?role .
        }
    """
    
    unified_graph = Graph()
    unified_graph += social_graph
    unified_graph += work_graph
    
    interest_results = unified_graph.query(interest_workplace_query)
    workplace_interests = {}
    
    for row in interest_results:
        person_name = str(row.person).split('/')[-1]
        interest_name = str(row.interest).split('/')[-1]
        workplace_name = str(row.workplace).split('/')[-1]
        role = str(row.role)
        
        if workplace_name not in workplace_interests:
            workplace_interests[workplace_name] = []
        workplace_interests[workplace_name].append(interest_name)
        
        print(f"  {person_name} ({role}) at {workplace_name} likes {interest_name}")
    
    # Workplace interest summary
    print("\nWorkplace interest patterns:")
    for workplace, interests in workplace_interests.items():
        unique_interests = set(interests)
        print(f"  {workplace}: {', '.join(unique_interests)}")
    
    # Performance benchmarking
    print("\n=== Performance Analysis ===")
    
    # Benchmark query performance
    simple_query = "SELECT ?s ?p ?o WHERE { ?s ?p ?o } LIMIT 10"
    
    start_time = time.time()
    results = list(unified_graph.query(simple_query))
    query_time = time.time() - start_time
    
    print(f"Query executed in {query_time:.6f} seconds")
    print(f"Retrieved {len(results)} results")
    
    # Graph size metrics
    print(f"\nGraph size metrics:")
    print(f"  Unified graph triples: {len(unified_graph)}")
    print(f"  Unique subjects: {len(set(unified_graph.subjects()))}")
    print(f"  Unique predicates: {len(set(unified_graph.predicates()))}")
    print(f"  Unique objects: {len(set(unified_graph.objects()))}")
    
    # Memory usage estimation
    import sys
    try:
        graph_size = sys.getsizeof(unified_graph)
        print(f"  Estimated graph memory: {graph_size} bytes")
    except:
        print("  Memory estimation not available")
        raise
    
    # Graph modification operations
    print("\n=== Graph Modification Operations ===")
    
    # Add new relationships dynamically
    test_graph = Graph()
    test_graph += unified_graph  # Copy existing data
    
    # Add new person and relationships
    eve = ex.eve
    test_graph.add((eve, social.knows, alice))
    test_graph.add((eve, social.likes, ex.rust))
    test_graph.add((eve, work.worksAt, ex.BlockchainCorp))
    test_graph.add((eve, work.hasRole, Literal("Blockchain Developer")))
    
    print(f"Original graph: {len(unified_graph)} triples")
    print(f"Modified graph: {len(test_graph)} triples")
    print(f"Added: {len(test_graph) - len(unified_graph)} new triples")
    
    # Remove specific relationships
    test_graph.remove((alice, social.knows, charlie))
    print(f"After removal: {len(test_graph)} triples")
    
    # Graph comparison
    common_triples = set(unified_graph) & set(test_graph)
    unique_to_original = set(unified_graph) - set(test_graph)
    unique_to_modified = set(test_graph) - set(unified_graph)
    
    print(f"\nGraph comparison:")
    print(f"  Common triples: {len(common_triples)}")
    print(f"  Unique to original: {len(unique_to_original)}")
    print(f"  Unique to modified: {len(unique_to_modified)}")
    
    # Transaction-like operations
    print("\n=== Batch Operations ===")
    
    batch_graph = Graph()
    
    # Simulate batch insertion
    batch_data = [
        (ex.person1, social.knows, ex.person2),
        (ex.person1, social.likes, ex.topic1),
        (ex.person2, social.likes, ex.topic2),
        (ex.person1, work.worksAt, ex.company1),
        (ex.person2, work.worksAt, ex.company2)
    ]
    
    start_time = time.time()
    for triple in batch_data:
        batch_graph.add(triple)
    batch_time = time.time() - start_time
    
    print(f"Batch inserted {len(batch_data)} triples in {batch_time:.6f} seconds")
    print(f"Average time per triple: {batch_time/len(batch_data):.8f} seconds")
    
    print("Advanced RDF operations and graph analytics completed successfully")
    
except Exception as e:
    print(f"Error in advanced RDF operations: {e}")
    raise AssertionError(
        f"rdflib example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e